In [30]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import StratifiedKFold

df = pd.read_csv("dataset.csv")
df

,transaction_id,amount_usd,merchant_category,card_type,auth_method,channel,device_type,is_foreign_transaction,hours_since_last_txn,txn_count_last_24h,...,ip_country_mismatch,billing_shipping_mismatch,cvv_retry_count,velocity_score,time_of_day_hour,day_of_week,is_ai_generated_scam_attempt,merchant_risk_score,prior_disputes,is_fraud
0,1,42.86,Restaurants,Visa,OTP,Online,Android Phone,False,13.54,2,...,False,False,0,0.1,18,3,False,42.3,0,0
1,2,4.75,Online Retail,Mastercard,3D Secure,Online,Android Phone,False,0.71,2,...,False,False,0,25.8,12,4,False,28.3,0,0
2,3,77.18,Groceries,Mastercard,3D Secure,Online,Mac,False,0.35,5,...,False,True,0,42.3,5,0,False,24.7,1,0
3,4,1.69,Streaming,Visa,No Authentication,POS,Android Phone,False,3.42,6,...,False,False,0,28.9,22,6,False,56.2,1,0
4,5,261.68,Travel,Visa,3D Secure,In-App,iPhone,False,2.43,2,...,False,False,0,3.9,2,4,False,32.7,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,19996,41.40,Healthcare,Mastercard,OTP,POS,Android Phone,False,8.82,2,...,False,False,0,17.3,2,3,False,29.6,1,0
19996,19997,47.92,Online Retail,Visa,OTP,POS,Android Phone,False,45.81,1,...,False,False,0,9.0,4,0,False,29.2,1,0
19997,19998,10.25,Restaurants,Visa,3D Secure,In-App,Tablet,False,20.67,0,...,False,False,1,1.0,19,5,False,22.9,0,0
19998,19999,31.34,Travel,Visa,3D Secure,Online,Android Phone,False,22.94,2,...,False,False,0,8.8,2,1,False,27.9,0,0


In [31]:
for col in df.columns:
    print(f'\n{col}')
    print(df[col].unique()[:20])


transaction_id
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]

amount_usd
[  42.86    4.75   77.18    1.69  261.68  119.72  139.71  340.68  116.41
  154.65   64.78  166.57   37.45    6.08   21.76  229.32  305.01   86.01
 1055.74   47.59]

merchant_category
['Restaurants' 'Online Retail' 'Groceries' 'Streaming' 'Travel'
 'Gift Cards' 'Electronics' 'Fuel' 'Gaming' 'Utilities' 'Crypto Exchange'
 'Healthcare']

card_type
['Visa' 'Mastercard' 'Amex' 'RuPay' 'Discover']

auth_method
['OTP' '3D Secure' 'No Authentication' 'Biometric' 'PIN']

channel
['Online' 'POS' 'In-App' 'Contactless' 'ATM']

device_type
['Android Phone' 'Mac' 'iPhone' 'POS Terminal' 'ATM Machine' 'Tablet'
 'Windows PC' 'Smart Watch']

is_foreign_transaction
[False  True]

hours_since_last_txn
[13.54  0.71  0.35  3.42  2.43  5.94 18.31 25.56  6.28  0.78  0.32  5.78
 11.63  5.98  3.39  3.72  2.54  1.48  0.68  4.1 ]

txn_count_last_24h
[ 2  5  6  4  3  1  7  0  8  9 10 11 12]

distance_from_home_km
[22.35 35.

In [32]:
# Features Engineering

df['log_amount_usd'] = np.log1p(df['amount_usd'])
df['cumulitative_risk_factor'] = df[['used_vpn', "ip_country_mismatch", "billing_shipping_mismatch", "is_foreign_transaction", "is_ai_generated_scam_attempt", "is_new_merchant"]].sum(axis=1)
df['amount_to_balance_ratio'] = df['amount_usd'] / (df['account_balance_usd'] + 1)
df['velocity_to_gap_ratio'] = df["velocity_score"] / (df['hours_since_last_txn'] + 0.1)
merchant_categories = ['Crypto Exchange', 'Gift Cards', 'Gaming']
df['is_high_risk_merchant'] = df['merchant_category'].isin(merchant_categories)
df['is_aunthenticated'] = df['auth_method'] == "No Authentication"
df.columns

Index(['transaction_id', 'amount_usd', 'merchant_category', 'card_type',
       'auth_method', 'channel', 'device_type', 'is_foreign_transaction',
       'hours_since_last_txn', 'txn_count_last_24h', 'distance_from_home_km',
       'card_age_months', 'customer_age', 'account_balance_usd',
       'is_new_merchant', 'used_vpn', 'ip_country_mismatch',
       'billing_shipping_mismatch', 'cvv_retry_count', 'velocity_score',
       'time_of_day_hour', 'day_of_week', 'is_ai_generated_scam_attempt',
       'merchant_risk_score', 'prior_disputes', 'is_fraud', 'log_amount_usd',
       'cumulitative_risk_factor', 'amount_to_balance_ratio',
       'velocity_to_gap_ratio', 'is_high_risk_merchant', 'is_aunthenticated'],
      dtype='object')

In [33]:
negative = (df["is_fraud"] == 0).sum()
positive = (df["is_fraud"] == 1).sum()
scale_pos_weight = negative / positive
print(scale_pos_weight)


57.99705014749262


In [ ]:
X = df.drop(columns=['transaction_id', 'is_fraud', "amount_usd", "account_balance_usd", "velocity_score"])
y = df['is_fraud']

skf = StratifiedKFold(n_splits =5, shuffle=True, random_state=42)
for fold, (train_idx, valid_idx) in enumerate(
    skf.split(X, y), 1
):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    print(
        f"Fold {fold}:",
        y_train.mean(),
        y_valid.mean()
    )



Fold 1: 0.017 0.01675
Fold 2: 0.0169375 0.017
Fold 3: 0.0169375 0.017
Fold 4: 0.0169375 0.017
Fold 5: 0.0169375 0.017


In [ ]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

In [ ]:
from catboost import CatBoostClassifier

cat_features = [
    "merchant_category",
    "card_type",
    "auth_method",
    "channel",
    "device_type",
    "day_of_week"
]

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="PRAUC",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=False
)

In [ ]:
from sklearn.linear_model import LogisticRegression


ct = ColumnTransformer

lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

In [ ]:
from imblearn.ensemble import BalancedRandomForestClassifier

brf = BalancedRandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

In [ ]:
from sklearn.metrics import average_precision_score
probabilities = model.predict_proba(X_valid)[:, 1]

pr_auc = average_precision_score(
    y_valid,
    probabilities
)

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(
    y_valid,
    probabilities
)

valid = precision[:-1] >= 0.85

if valid.any():
    recall_at_85_precision = recall[:-1][valid].max()
else:
    recall_at_85_precision = 0

In [ ]:
from sklearn.metrics import fbeta_score

f2 = fbeta_score(
    y_valid,
    predictions,
    beta=2
)

In [ ]:
thresholds = np.arange(
    0.01,
    1.00,
    0.01
)
predictions = (
    probabilities >= threshold
).astype(int)